# 04 · User Segmentation

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Objective

This notebook identifies user segments with the highest onboarding friction and the highest opportunity for improvement.

Segmentation is focused on business actionability: channel, device, region, customer type, account profile, document readiness and business-hour behavior.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users.head()

## 2. Segment Scorecard

In [ ]:
users_with_support = users.copy()
users_with_support["has_support_call"] = users_with_support["user_id"].isin(support["user_id"]).astype(int)
users_with_support["upload_now"] = users_with_support["upload_choice"].eq("upload_now").astype(int)

def segment_scorecard(cols):
    return (
        users_with_support
        .groupby(cols)
        .agg(
            users=("user_id", "count"),
            approval_rate=("approved", "mean"),
            upload_now_rate=("upload_now", "mean"),
            document_submission_rate=("document_submitted", "mean"),
            support_rate=("has_support_call", "mean"),
            avg_minutes_to_upload=("minutes_to_upload", "mean")
        )
        .reset_index()
        .sort_values(["approval_rate", "users"], ascending=[True, False])
    )

channel_scorecard = segment_scorecard(["channel"])
channel_scorecard

## 3. Segment Opportunity Index

In [ ]:
score = channel_scorecard.copy()
score["approval_gap"] = users["approved"].mean() - score["approval_rate"]
score["support_gap"] = score["support_rate"] - users_with_support["has_support_call"].mean()
score["opportunity_index"] = (
    score["users"] / score["users"].sum()
    * (score["approval_gap"].clip(lower=0) + score["support_gap"].clip(lower=0))
)
score.sort_values("opportunity_index", ascending=False)

In [ ]:
top = score.sort_values("opportunity_index", ascending=False).set_index("channel")["opportunity_index"]
ax = top.plot(kind="bar", figsize=(10, 5))
ax.set_title("Opportunity Index by Acquisition Channel")
ax.set_ylabel("Opportunity index")
ax.set_xlabel("Channel")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 4. Multi-dimensional Segments

In [ ]:
multi = segment_scorecard(["channel", "device", "customer_type"])
multi = multi[multi["users"] >= 500].copy()
multi["approval_gap"] = users["approved"].mean() - multi["approval_rate"]
multi["opportunity_index"] = multi["users"] / len(users) * multi["approval_gap"].clip(lower=0)
multi.sort_values("opportunity_index", ascending=False).head(15)

## 5. Document Readiness

In [ ]:
readiness = segment_scorecard(["has_document_ready"])
readiness

In [ ]:
readiness_plot = readiness.set_index("has_document_ready")[["upload_now_rate", "approval_rate", "support_rate"]]
ax = readiness_plot.plot(kind="bar", figsize=(8, 5))
ax.set_title("Impact of Document Readiness")
ax.set_ylabel("Rate")
ax.set_xlabel("Has document ready")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Actionable Segment Recommendations

In [ ]:
recommendations = pd.DataFrame({
    "segment_theme": [
        "Paid acquisition users",
        "Users without documents ready",
        "Upload-later users",
        "High-support segments",
        "Low-approval device/channel combinations"
    ],
    "recommended_action": [
        "Improve expectation-setting before onboarding starts",
        "Add pre-onboarding checklist and reminder nudges",
        "Redesign document screen to make immediate upload the default path",
        "Create proactive in-app guidance before support contact",
        "Prioritize UX QA and device-specific friction analysis"
    ],
    "expected_business_effect": [
        "Higher downstream conversion quality",
        "Higher upload-now rate",
        "Lower abandonment at document step",
        "Reduced support cost",
        "Improved approval conversion"
    ]
})
recommendations